In [2]:
from pathlib import Path
import numpy as np
from typing import Literal, Optional
import os
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
import seaborn as sns


project_root = Path(".").resolve().parent.parent.parent
assert project_root.name == "pep-compass"

In [3]:
import pandas as pd

parents_negative = pd.read_csv(
    project_root
    / "results/data/veltri_negative.csv"
)
mutants_negative = pd.read_csv(
    project_root
    / "results/data/mutants/veltri_negative/veltri_negative_direction_threshold=0.001_token_threshold=0.1_jacobian_mode=approx.csv"
)

parents_positive = pd.read_csv(
    project_root
    / "results/data/veltri_positive.csv"
)
mutants_positive = pd.read_csv(
    project_root
    / "results/data/mutants/veltri_positive/veltri_positive_direction_threshold=0.001_token_threshold=0.1_jacobian_mode=approx.csv"
    )

In [4]:
def compute_diff(
    parents_df: pd.DataFrame,
    mutants_df: pd.DataFrame,
    match_col_parents: str,
    match_col_mutants: str,
    value_cols: list[str],
    value_preprocessing: Optional[Literal["log"]],
    add_relative: bool,
) -> pd.DataFrame:
    """
    Compute the difference between the mutants and parents for each value column.
    """
    merged = mutants_df.merge(
        parents_df,
        left_on=match_col_mutants,
        right_on=match_col_parents,
        suffixes=("_mutants", "_parents"),
    )
    if value_preprocessing is None:
        vprep_fn = lambda x: x
    elif value_preprocessing == "log":
        vprep_fn = np.log2
    else:
        raise ValueError(f"Invalid value_preprocessing: {value_preprocessing}")
    if value_preprocessing is None:
        value_preprocessing = ""
    else:
        value_preprocessing = "_" + value_preprocessing

    for value_col in value_cols:
        merged[f"{value_col}{value_preprocessing}_diff"] = vprep_fn(
            merged[f"{value_col}_mutants"]
        ) - vprep_fn(merged[f"{value_col}_parents"])
        if add_relative:
            merged[f"{value_col}{value_preprocessing}_diff_relative"] = merged[
                f"{value_col}{value_preprocessing}_diff"
            ] / vprep_fn(merged[f"{value_col}_parents"])
    return merged


In [6]:
mic_bac_columns = parents_negative.head().columns.tolist()[2:]

In [7]:
diff_negative = compute_diff(
    parents_df=parents_negative,
    mutants_df=mutants_negative,
    match_col_parents="Name",
    match_col_mutants="parent_name",
    value_cols=mic_bac_columns,
    value_preprocessing="log",
    add_relative=True,
)
diff_positive = compute_diff(
    parents_df=parents_positive,
    mutants_df=mutants_positive,
    match_col_parents="Name",
    match_col_mutants="parent_name",
    value_cols=mic_bac_columns,
    value_preprocessing="log",
    add_relative=True,
)

ALL_AA = list("ACDEFGHIKLMNPQRSTVWY")

diff_clean_positive = diff_positive.dropna(subset=["parent", "mutant", "position"]).drop_duplicates(subset=["parent", "mutant", "position"])


mask = (
    (diff_clean_positive["position"] < diff_clean_positive["parent"].str.len()) &
    (diff_clean_positive["position"] < diff_clean_positive["mutant"].str.len())
)
diff_clean_positive = diff_clean_positive[mask]


diff_clean_negative = diff_negative.dropna(subset=["parent", "mutant", "position"]).drop_duplicates(subset=["parent", "mutant", "position"])


mask = (
    (diff_clean_negative["position"] < diff_clean_negative["parent"].str.len()) &
    (diff_clean_negative["position"] < diff_clean_negative["mutant"].str.len())
)
diff_clean_negative = diff_clean_negative[mask]

In [8]:
#merge diffs into one but leave label 0 negative 1 postive
diff_clean_negative['label'] = 0
diff_clean_positive['label'] = 1
diff_clean = pd.concat([diff_clean_negative, diff_clean_positive], ignore_index=True)



In [9]:
diff_clean

,mutant,position,parent,direction_significance_threshold,token_threshold,parent_name,parent_sequence,A. baumannii ATCC 19606_mutants,E. coli ATCC 11775_mutants,E. coli AIG221_mutants,...,E. coli Nissle_log_diff_relative,Salmonella enterica ATCC 9150 (BEIRES NR-515)_log_diff,Salmonella enterica ATCC 9150 (BEIRES NR-515)_log_diff_relative,Salmonella enterica (BEIRES NR-170)_log_diff,Salmonella enterica (BEIRES NR-170)_log_diff_relative,Salmonella enterica ATCC 9150 (BEIRES NR-174)_log_diff,Salmonella enterica ATCC 9150 (BEIRES NR-174)_log_diff_relative,L. monocytogenes ATCC 19111 (BEIRES NR-106)_log_diff,L. monocytogenes ATCC 19111 (BEIRES NR-106)_log_diff_relative,label
0,EGAVSGVEGLPSGSAL,15,EGAVSGVEGLPSGSAL,0.001,0.1,UniRef50_A0QYG2,EGAVSGVEGLPSGSAL,138.511200,139.801180,137.902370,...,0.000000,4.745545e-07,7.047766e-08,6.525858e-07,6.454656e-08,-1.338066e-07,-1.433012e-08,0.000000,0.000000,0
1,EGAVSGVEGLPSGSAN,15,EGAVSGVEGLPSGSAL,0.001,0.1,UniRef50_A0QYG2,EGAVSGVEGLPSGSAL,139.289950,140.260880,138.068270,...,0.003851,3.233473e-02,4.802137e-03,-1.855647e-02,-1.835401e-03,6.475522e-02,6.935012e-03,0.021393,0.003040,0
2,EGAVSGVEGLPSGSAK,15,EGAVSGVEGLPSGSAL,0.001,0.1,UniRef50_A0QYG2,EGAVSGVEGLPSGSAL,140.665590,141.033480,138.658480,...,-0.002814,2.686084e-02,3.989192e-03,-2.276042e-02,-2.251209e-03,3.817424e-02,4.088301e-03,0.000285,0.000041,0
3,EGAVSGVEGLPSGSAL,7,EGAVSGVEGLPSGSAL,0.001,0.1,UniRef50_A0QYG2,EGAVSGVEGLPSGSAL,138.511200,139.801180,137.902370,...,0.000000,4.745545e-07,7.047766e-08,6.525858e-07,6.454656e-08,-1.338066e-07,-1.433012e-08,0.000000,0.000000,0
4,EGAVSGVLGLPSGSAL,7,EGAVSGVEGLPSGSAL,0.001,0.1,UniRef50_A0QYG2,EGAVSGVEGLPSGSAL,137.831100,138.915990,137.481280,...,-0.001433,5.090639e-03,7.560277e-04,2.073769e-04,2.051143e-05,-2.071475e-02,-2.218462e-03,-0.016658,-0.002367,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143218,FLNALKNFAKTAGYRLKSLLN,13,FLNALKNFAKTAGKRLKSLLN,0.001,0.1,AP02211,FLNALKNFAKTAGKRLKSLLN,58.525154,88.008960,75.385345,...,0.067512,7.500116e-02,1.389157e-02,6.910766e-01,7.924917e-02,3.307591e-01,4.070951e-02,0.054493,0.009464,1
143219,FLKALKNFAKTAGKRLKSLLN,2,FLNALKNFAKTAGKRLKSLLN,0.001,0.1,AP02211,FLNALKNFAKTAGKRLKSLLN,50.411660,60.938366,56.898110,...,0.012493,-1.428481e-01,-2.645805e-02,6.125704e-02,7.024648e-03,6.698837e-02,8.244863e-03,-0.056720,-0.009851,1
143220,FLLALKNFAKTAGKRLKSLLN,2,FLNALKNFAKTAGKRLKSLLN,0.001,0.1,AP02211,FLNALKNFAKTAGKRLKSLLN,46.594486,70.239750,61.421112,...,-0.003623,-1.057700e-01,-1.959052e-02,-3.537130e-03,-4.056201e-04,1.106455e-01,1.361814e-02,-0.095709,-0.016623,1
143221,FLNALKKFAKTAGKRLKSLLN,6,FLNALKNFAKTAGKRLKSLLN,0.001,0.1,AP02211,FLNALKNFAKTAGKRLKSLLN,51.353294,61.891838,58.844350,...,-0.032869,-1.472630e-01,-2.727578e-02,-1.301472e-01,-1.492463e-02,-1.745845e-01,-2.148769e-02,-0.132572,-0.023025,1


In [ ]:
diff_clean['transition'] =diff_clean.apply(lambda row: row['mutant'][int(row['position'])] + row['parent'][int(row['position'])], axis=1)

In [17]:
y = diff_clean['E. coli ATCC 11775_mutants']

In [33]:
transition_dummies = pd.get_dummies(diff_clean['transition'], prefix='trans', drop_first=True)


In [35]:
transition_dummies

,trans_AC,trans_AD,trans_AE,trans_AF,trans_AG,trans_AH,trans_AI,trans_AK,trans_AL,trans_AM,...,trans_YM,trans_YN,trans_YP,trans_YQ,trans_YR,trans_YS,trans_YT,trans_YV,trans_YW,trans_YY
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143218,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
143219,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
143220,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
143221,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
# transition_dummies = transition_dummies.mul(diff_clean['label'], axis=0)


In [34]:
from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd
import statsmodels.api as sm

# X and y
X = transition_dummies.values  # no transpose
y = diff_clean['E. coli ATCC 11775_mutants']

# Fit OLS model
X_sm = sm.add_constant(X)  # Adds intercept
ols_model = sm.OLS(y, X_sm)
results = ols_model.fit()

# Extract coefficients & p-values (excluding constant)
model_coef_df = pd.DataFrame({
    "aa_change": transition_dummies.columns,
    "coefficient": results.params,  # skip constant
    "p_value": results.pvalues      # skip constant
})

# Display
print(model_coef_df.head())


TypeError: numpy boolean subtract, the `-` operator, is not supported, use the bitwise_xor, the `^` operator, or the logical_xor function instead.

In [29]:
from statsmodels.stats.multitest import multipletests

In [30]:
# --- Apply Benjamini–Hochberg correction (FDR)
reject, pvals_corrected, _, _ = multipletests(model_coef_df["p_value"], method="fdr_bh")

# Add to DataFrame
model_coef_df["p_adj"] = pvals_corrected
model_coef_df["significant_FDR_5%"] = reject

# --- Sort by adjusted p-value for clarity
model_coef_df = model_coef_df.sort_values("p_adj")

In [32]:
model_coef_df[model_coef_df["significant_FDR_5%"]!=True]

,aa_change,coefficient,p_value,p_adj,significant_FDR_5%
